<a href="https://colab.research.google.com/github/Rakshada1811/FlyRank-Assignment-Repo/blob/main/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rakshada1811/FlyRank-Assignment-Repo/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

### Baseline Rule

The baseline action score ranks content pages according to how likely they are to require attention.

The rule gives higher priority to pages that have:

- a large drop in impressions during the last 30 days compared to the previous 30 days,
- enough historical impressions to make the comparison meaningful,
- lower visibility across search queries,
- higher concentration of traffic from a small number of queries.

The score is intended for decision support only. It is a simple heuristic that helps identify pages that may require manual review before taking action.

### Reason Codes

| Reason Code | Meaning |
|--------------|-------------------------------------------------------------|
| IMP_DROP | Impressions decreased significantly compared with the previous month. |
| LOW_VISIBILITY | The page appears for only a small number of search queries. |
| HIGH_QUERY_CONCENTRATION | Most traffic comes from only a few queries, making performance less stable. |
| LOW_HISTORY | The page has limited historical impressions, so confidence is lower. |
| REVIEW | Manual review is recommended before making any changes. |

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Baseline rule defined successfully.")

reason_codes = [
    "IMP_DROP",
    "LOW_VISIBILITY",
    "HIGH_QUERY_CONCENTRATION",
    "LOW_HISTORY",
    "REVIEW"
]

print(reason_codes)

Baseline rule defined successfully.
['IMP_DROP', 'LOW_VISIBILITY', 'HIGH_QUERY_CONCENTRATION', 'LOW_HISTORY', 'REVIEW']


## 2. Build the ranked queue (writes the CSV)

### Baseline Action Score

A simple baseline action score is computed using the available features.

The score increases when:

- impressions have declined more sharply,
- the page ranks for fewer search queries,
- traffic is concentrated in only a few queries.

The ranked output is intended for decision support only. It provides a simple priority order for manual review rather than making automatic decisions.

In [9]:
# ==========================================
# Build model_data
# ==========================================

data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev30"]
).astype(int)

feature_cols = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

model_data = data.dropna(subset=feature_cols)

print(model_data.shape)
model_data.head()

(102203, 13)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share,is_declining
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159,15.0,0.144623,0.665019,79.0,308.0,0.256494,0
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091,101.0,0.037423,0.178737,15557.0,18432.0,0.844021,1
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206,3.0,0.215054,0.623656,25.0,60.0,0.416667,1
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655,16.0,0.032740,0.717915,473.0,952.0,0.496849,1
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483,8.0,0.224066,0.630705,30.0,140.0,0.214286,0


In [12]:
import os
import numpy as np

baseline = model_data.copy()

# Percentage decline in impressions
baseline["decline_pct"] = (
    (baseline["imp_prev30"] - baseline["imp_last30"])
    / baseline["imp_prev30"]
)

baseline["decline_pct"] = baseline["decline_pct"].clip(lower=0)

# Simple baseline score
baseline["baseline_action_score"] = (
    baseline["decline_pct"] * 60
    + (1 / (baseline["visible_queries"] + 1)) * 20
    + baseline["top_query_share"] * 20
)

baseline["reason_code"] = np.where(
    baseline["decline_pct"] > 0.20,
    "IMP_DROP",
    "REVIEW"
)

baseline = baseline.sort_values(
    "baseline_action_score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

os.makedirs("work/outputs", exist_ok=True)

baseline.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Rows:", len(baseline))
print("Saved successfully.")
print()

baseline[
    [
        "rank",
        "content_hash_id",
        "baseline_action_score",
        "reason_code"
    ]
].head(20)

Rows: 102203
Saved successfully.



,rank,content_hash_id,baseline_action_score,reason_code
0,1,content_b066218f63506e43,90.0,IMP_DROP
1,2,content_89ae1ba0081e0515,90.0,IMP_DROP
2,3,content_b103f45f7635510d,90.0,IMP_DROP
3,4,content_12f8f5af4574bead,90.0,IMP_DROP
4,5,content_70e83bec91c206b8,90.0,IMP_DROP
5,6,content_4bae68c8289a9a88,90.0,IMP_DROP
6,7,content_593e7b6e791322c1,90.0,IMP_DROP
7,8,content_db17fe62c9ed54f2,90.0,IMP_DROP
8,9,content_0482ee768bb8de83,90.0,IMP_DROP
9,10,content_335c9eb889ed4f99,90.0,IMP_DROP


In [10]:
print("con:", "con" in globals())
print("TABLES:", "TABLES" in globals())
print("features:", "features" in globals())
print("data:", "data" in globals())
print("model_data:", "model_data" in globals())

if "model_data" in globals():
    print(model_data.columns.tolist())

con: True
TABLES: True
features: True
data: True
model_data: True
['client_hash_id', 'content_hash_id', 'imp_last30', 'imp_prev30', 'clk_last30', 'pos_last30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_impressions', 'kept_impressions', 'top_query_share', 'is_declining']


In [4]:
# ==========================================
# Build baseline dataset from full warehouse
# ==========================================

%pip -q install duckdb huggingface_hub

import duckdb
import pandas as pd
import numpy as np
import os
import getpass

# -----------------------
# HuggingFace Token
# -----------------------
HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your HuggingFace READ token: ")

# -----------------------
# Connect DuckDB
# -----------------------
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily":
    f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_query":
    f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("Connected successfully.")

Paste your HuggingFace READ token: ··········
Connected successfully.


In [6]:
# ==========================================
# Build feature table
# ==========================================

features = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
),

windowed AS (

SELECT
    f.client_hash_id,
    f.content_hash_id,

    SUM(
        CASE
            WHEN report_date > end_d - INTERVAL 30 DAY
            THEN gsc_impressions
            ELSE 0
        END
    ) AS imp_last30,

    SUM(
        CASE
            WHEN report_date <= end_d - INTERVAL 30 DAY
            THEN gsc_impressions
            ELSE 0
        END
    ) AS imp_prev30,

    SUM(
        CASE
            WHEN report_date > end_d - INTERVAL 30 DAY
            THEN gsc_clicks
            ELSE 0
        END
    ) AS clk_last30,

    AVG(
        CASE
            WHEN report_date > end_d - INTERVAL 30 DAY
            THEN gsc_avg_position
        END
    ) AS pos_last30

FROM {TABLES['fact_daily']} f,
     bounds

WHERE report_date > end_d - INTERVAL 60 DAY

GROUP BY
    client_hash_id,
    content_hash_id

HAVING imp_prev30 >= 100

)

SELECT *
FROM windowed

""").df()

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(111247, 6)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483


In [8]:
# ==========================================
# Add query-level signals
# ==========================================

qsignals = con.sql(f"""
SELECT
    content_hash_id,
    ANY_VALUE(content_visible_query_count) AS visible_queries,
    ANY_VALUE(rare_impressions_share) AS rare_share,
    ANY_VALUE(anonymized_impressions_share) AS anon_share,
    MAX(impressions_90d) AS top_query_impressions,
    SUM(impressions_90d) AS kept_impressions
FROM {TABLES['fact_query']}
GROUP BY content_hash_id
""").df()

qsignals["top_query_share"] = (
    qsignals["top_query_impressions"] /
    qsignals["kept_impressions"]
)

data = features.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

print(data.shape)
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(111247, 12)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_62f4a7e64f5e0096,content_0dac238195631de4,219.0,251.0,0.0,3.468159,15.0,0.144623,0.665019,79.0,308.0,0.256494
1,client_62f4a7e64f5e0096,content_f6116743b00afc2d,995.0,4786.0,2.0,25.024091,101.0,0.037423,0.178737,15557.0,18432.0,0.844021
2,client_62f4a7e64f5e0096,content_332f337f995e9781,103.0,177.0,0.0,18.600206,3.0,0.215054,0.623656,25.0,60.0,0.416667
3,client_62f4a7e64f5e0096,content_82053c8b9d8a4811,736.0,1518.0,2.0,9.526655,16.0,0.032740,0.717915,473.0,952.0,0.496849
4,client_62f4a7e64f5e0096,content_742939be32c206d2,269.0,335.0,3.0,7.904483,8.0,0.224066,0.630705,30.0,140.0,0.214286


## 3. Top-20 review

The highest ranked pages are reviewed manually after the baseline score is calculated.

For each page, the following information is reported:

- Recommended action
- Reason code
- Confidence note
- What could make the prediction incorrect

The review is intended for decision support only. The final decision should always be made after manual inspection.

In [15]:
top20 = baseline.head(20).copy()

top20["action"] = "Review page"

top20["confidence_note"] = np.where(
    top20["baseline_action_score"] >= 60,
    "High confidence",
    "Medium confidence"
)

top20["what_could_be_wrong"] = (
    "Performance may be affected by seasonality, recent updates, "
    "or temporary search trends."
)

review = top20[
    [
        "rank",
        "content_hash_id",
        "baseline_action_score",
        "action",
        "reason_code",
        "confidence_note",
        "what_could_be_wrong"
    ]
]

print(review)

review.to_csv(
    "work/outputs/top20_review.csv",
    index=False
)

print("\nTop-20 review saved successfully.")

    rank           content_hash_id  baseline_action_score       action  \
0      1  content_b066218f63506e43                   90.0  Review page   
1      2  content_89ae1ba0081e0515                   90.0  Review page   
2      3  content_b103f45f7635510d                   90.0  Review page   
3      4  content_12f8f5af4574bead                   90.0  Review page   
4      5  content_70e83bec91c206b8                   90.0  Review page   
5      6  content_4bae68c8289a9a88                   90.0  Review page   
6      7  content_593e7b6e791322c1                   90.0  Review page   
7      8  content_db17fe62c9ed54f2                   90.0  Review page   
8      9  content_0482ee768bb8de83                   90.0  Review page   
9     10  content_335c9eb889ed4f99                   90.0  Review page   
10    11  content_44da9c95bc40accf                   90.0  Review page   
11    12  content_5bf031d32530681d                   90.0  Review page   
12    13  content_5135d56282ba5e85    

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some pages may receive a high baseline score even though their decline is temporary or caused by external factors such as seasonality, marketing campaigns, or search demand changes.

The baseline score is intended for prioritization rather than automatic decision making. Manual review is recommended before taking action.

No product-specific flags, client identifiers, or future information were used while calculating the score. The ranking was generated only from historical observations available before the decision point, reducing the risk of data leakage.

In [16]:
weak_picks = baseline[
    baseline["baseline_action_score"] < baseline["baseline_action_score"].median()
].head(10)

print("Sample weak picks")
print(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "baseline_action_score",
            "reason_code"
        ]
    ]
)

print("\nLeakage Check")
print("----------------------------")
print("✓ No product flags used")
print("✓ No future dates used")
print("✓ Score calculated only from historical features")
print("✓ Intended for decision support only")

Sample weak picks
        rank           content_hash_id  baseline_action_score reason_code
51102  51103  content_695b03bab85aba92              31.430132    IMP_DROP
51103  51104  content_967187171606f368              31.428958      REVIEW
51104  51105  content_2227165d541f6531              31.428105    IMP_DROP
51105  51106  content_5e1ef2d473c84432              31.427416    IMP_DROP
51106  51107  content_5ad08a8d3220207d              31.427337    IMP_DROP
51107  51108  content_00a22d5328b0d03b              31.427288    IMP_DROP
51108  51109  content_1ce95960958e86b4              31.426631    IMP_DROP
51109  51110  content_74bb9805d8aa013f              31.426229    IMP_DROP
51110  51111  content_06e82a3eb58416bb              31.425963    IMP_DROP
51111  51112  content_da15d28942dd45b2              31.424989    IMP_DROP

Leakage Check
----------------------------
✓ No product flags used
✓ No future dates used
✓ Score calculated only from historical features
✓ Intended for decision supp

## Self-check

Before you submit, confirm each line honestly:

- ✓ Every section above is filled — markdown thinking AND the code that backs it
- ✓ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✓ No client names, URLs, or private queries anywhere
- ✓ My claims use careful words: observed, measured, directional, decision-support
- ✓ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.